<a href="https://colab.research.google.com/github/Megh-05/Data-Science/blob/main/Exp_No_9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import tensorflow as tf
import numpy as np

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (
    Input,
    Dense,
    Flatten,
    BatchNormalization,
    Lambda
)

# Configuration

feature_size = 32

image_dimension = (28, 28)

# Create Feature Extraction Network

def build_feature_model():

    image_input = Input(shape=image_dimension)

    layer = Flatten()(image_input)

    layer = Dense(
        256,
        activation='relu'
    )(layer)

    layer = BatchNormalization()(layer)

    layer = Dense(
        128,
        activation='relu'
    )(layer)

    final_vector = Dense(feature_size)(layer)

    feature_model = Model(
        image_input,
        final_vector
    )

    return feature_model

# Shared Feature Network

shared_network = build_feature_model()

# Define Three Inputs

base_image = Input(shape=image_dimension)

similar_image = Input(shape=image_dimension)

different_image = Input(shape=image_dimension)

# Generate Embeddings

base_vector = shared_network(base_image)

similar_vector = shared_network(similar_image)

different_vector = shared_network(different_image)

# Distance Calculation Function

def calculate_distance(data):

    first, second = data

    return tf.reduce_mean(
        tf.square(first - second),
        axis=1,
        keepdims=True
    )

# Positive Pair Distance

same_distance = Lambda(
    calculate_distance
)(
    [base_vector, similar_vector]
)

# Negative Pair Distance

diff_distance = Lambda(
    calculate_distance
)(
    [base_vector, different_vector]
)

# Distance Difference

triplet_output = Lambda(
    lambda values: values[0] - values[1]
)(
    [same_distance, diff_distance]
)

# Create Complete Triplet Model

embedding_model = Model(
    inputs=[
        base_image,
        similar_image,
        different_image
    ],
    outputs=triplet_output
)

# Custom Triplet Loss Function

def custom_triplet_loss(true_label, predicted_output):

    safe_margin = 0.8

    loss_value = tf.maximum(
        predicted_output + safe_margin,
        0.0
    )

    return tf.reduce_mean(loss_value)

# Compile Model

embedding_model.compile(
    optimizer='adam',
    loss=custom_triplet_loss
)

# Generate Random Training Data

samples = 120

anchor_images = np.random.rand(
    samples,
    28,
    28
)

positive_images = np.random.rand(
    samples,
    28,
    28
)

negative_images = np.random.rand(
    samples,
    28,
    28
)

target_output = np.zeros((samples, 1))

# Train Triplet Network

training_process = embedding_model.fit(
    [
        anchor_images,
        positive_images,
        negative_images
    ],
    target_output,
    epochs=8,
    batch_size=8
)

print("\nEmbedding Network Training Completed Successfully")

Epoch 1/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 5s 7ms/step - loss: 0.8159
Epoch 2/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 8ms/step - loss: 0.1377
Epoch 3/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0105
Epoch 4/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0020
Epoch 5/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0099
Epoch 6/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0096
Epoch 7/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0026
Epoch 8/8
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step - loss: 0.0026

Embedding Network Training Completed Successfully
